# Laboratorio · Fundamentos de machine learning

Vamos a predecir cuántas bicicletas se rentan cada día en Washington D. C. (dataset Bike Sharing de UCI, 2011 y 2012) a partir del clima y el calendario. Es el mismo tipo de problema que pronosticar demanda de producto, de refacciones o de market share: una cantidad que cambia en el tiempo.

En cada nivel escribe tu predicción antes de correr la celda. Las que fallan son las que más enseñan.

## Preparación

In [ ]:
import sys
sys.path.append("../src")          # el paquete del repo; en Colab usa: !pip install -q git+https://github.com/nielspac177/ml-fundamentos-lab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

from ml_fundamentos import (cargar_bike, split_temporal, baseline_ingenuo, metricas_regresion,
                            detectar_fuga, tabla_umbral)

df = cargar_bike("../data/bike_day.csv")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df["fecha"], df["viajes"], lw=0.8)
ax.axvline(pd.Timestamp("2012-07-01"), color="k", ls="--")
ax.set_title("Viajes por día · la línea marca dónde empieza el test")
plt.show()

Entrenamos con todo lo anterior al 1 de julio de 2012 y evaluamos con el segundo semestre de 2012. Nunca al revés: el modelo no puede ver el futuro.

In [ ]:
train, test = split_temporal(df, "2012-07-01")
VARIABLES = ["temporada", "anio", "mes", "feriado", "dia_semana", "dia_laboral",
             "clima", "temperatura", "sensacion_termica", "humedad", "viento"]
print(len(train), "días de train ·", len(test), "días de test")

## Nivel 1 · Primero, algo que cualquier modelo debe superar

**Predice antes de correr.** El baseline predice cada día con lo que pasó el mismo día de la semana anterior. ¿De cuántos viajes será su error promedio (MAE)?

_Tu predicción:_ 

In [ ]:
pred_base = baseline_ingenuo(df["viajes"], rezago=7).loc[test.index]
base = metricas_regresion(test["viajes"], pred_base)
base

**Explica.** ¿Por qué conviene calcular esto antes de entrenar cualquier modelo?

_Tu explicación:_ 

## Nivel 2 · Una regresión lineal

**Predice antes de correr.** ¿La regresión lineal con clima y calendario le gana al baseline? ¿Por cuánto?

_Tu predicción:_ 

In [ ]:
lineal = LinearRegression().fit(train[VARIABLES], train["viajes"])
m_lineal = metricas_regresion(test["viajes"], lineal.predict(test[VARIABLES]))
pd.DataFrame({"baseline": base, "lineal": m_lineal}).round(3)

In [ ]:
pd.Series(lineal.coef_, index=VARIABLES).sort_values().round(1)

**Explica.** El coeficiente de temperatura sale negativo y el de sensación térmica muy positivo. ¿Más calor reduce los viajes?

_Tu explicación:_ 

## Nivel 3 · El modelo perfecto que no sirve

**Predice antes de correr.** Agregamos dos columnas más: casuales y registrados. ¿Qué pasa con el R²?

_Tu predicción:_ 

In [ ]:
con_fuga = VARIABLES + ["casuales", "registrados"]
modelo_fuga = LinearRegression().fit(train[con_fuga], train["viajes"])
metricas_regresion(test["viajes"], modelo_fuga.predict(test[con_fuga]))

In [ ]:
detectar_fuga(df, "viajes")

**Explica.** ¿Por qué ese R² no sirve aunque sea casi 1? ¿Qué columnas de tu proyecto podrían tener el mismo problema?

_Tu explicación:_ 

## Nivel 4 · Sobreajuste

**Predice antes de correr.** Un árbol de decisión con más profundidad se ajusta mejor al train. ¿Qué crees que pasa con el error en test?

_Tu predicción:_ 

In [ ]:
filas = []
for prof in range(1, 21):
    arbol = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(train[VARIABLES], train["viajes"])
    filas.append({"profundidad": prof,
                  "MAE train": metricas_regresion(train["viajes"], arbol.predict(train[VARIABLES]))["MAE"],
                  "MAE test": metricas_regresion(test["viajes"], arbol.predict(test[VARIABLES]))["MAE"]})
curva = pd.DataFrame(filas).set_index("profundidad")
curva.plot(figsize=(8, 3.5), marker="o", title="Error contra profundidad del árbol")
plt.show()
curva.idxmin()

**Explica.** ¿En qué profundidad elegirías el árbol? ¿Por qué no en la que tiene menor error de train, ni en la de menor error de test?

_Tu explicación:_ 

## Nivel 5 · Cómo validas cambia lo que crees

**Predice antes de correr.** Medimos el árbol de profundidad 6 con validación cruzada de dos formas. ¿Cuál da un error más optimista: k-fold mezclado o validación temporal?

_Tu predicción:_ 

In [ ]:
arbol6 = DecisionTreeRegressor(max_depth=6, random_state=0)
mezclado = -cross_val_score(arbol6, train[VARIABLES], train["viajes"],
                            cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error")
temporal = -cross_val_score(arbol6, train[VARIABLES], train["viajes"],
                            cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error")
real = metricas_regresion(test["viajes"], arbol6.fit(train[VARIABLES], train["viajes"]).predict(test[VARIABLES]))["MAE"]
print(f"k-fold mezclado: {mezclado.mean():7.0f}")
print(f"temporal:        {temporal.mean():7.0f}")
print(f"test real:       {real:7.0f}")

**Explica.** ¿Qué validación usarías para los 32 meses del proyecto de Market share? ¿Y para el MRP?

_Tu explicación:_ 

## Nivel 6 · De predecir cuánto a decidir sí o no

Un día es de **caída fuerte** si se rentan menos del 70% de lo normal reciente (la mediana de las 4 semanas anteriores). Queremos avisarlo con anticipación para no sobreabastecer estaciones. La etiqueta solo usa información del pasado.

**Predice antes de correr.** Con el umbral por defecto de 0.5, ¿qué porcentaje de las caídas crees que detecta el modelo (recall)?

_Tu predicción:_ 

In [ ]:
df["nivel_reciente"] = df["viajes"].shift(1).rolling(28).median()     # solo días anteriores
df["caida"] = (df["viajes"] < 0.7 * df["nivel_reciente"]).astype(int)
d = df.dropna(subset=["nivel_reciente"])
tr, te = split_temporal(d, "2012-07-01")
logit = LogisticRegression(max_iter=3000).fit(tr[VARIABLES], tr["caida"])
score = logit.predict_proba(te[VARIABLES])[:, 1]
print(f"días con caída en test: {te['caida'].mean():.0%}")
tabla = tabla_umbral(te["caida"], score, umbrales=[0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
tabla.round(2)

**Predice antes de correr.** No detectar una caída (FN) cuesta 5 veces más que una falsa alarma (FP). ¿Qué umbral eliges?

_Tu predicción:_ 

In [ ]:
tabla["costo"] = 5 * tabla["FN"] + 1 * tabla["FP"]
tabla.sort_values("costo").head(3).round(2)

**Explica.** ¿Por qué el mejor umbral casi nunca es 0.5?

_Tu explicación:_ 

## Para tu proyecto

Responde en equipo, con tus datos:

1. ¿Cuál es tu variable objetivo y a qué granularidad (día, mes, material y CEDIS)?
2. ¿Cuántas observaciones independientes tienes de verdad? Millones de filas pueden ser 32 puntos después de agregar.
3. ¿Cuál es tu baseline?
4. ¿Qué columnas solo se conocen después del momento de predecir?
5. ¿Cómo vas a partir train y test en el tiempo?